# Sistema inteligente para la detección de anomalías con SAM 1 + PatchCore en MVTec AD

Cuaderno experimental de SamCore. Para cada categoría de objetos de MVTec AD
compara PatchCore sobre la imagen completa (línea base) contra PatchCore sobre
la región de interés obtenida con SAM (segmentación automática con pesos
congelados, regla de selección fijada a priori y caja cuadrada con margen), y
exporta los artefactos que consume la aplicación web.

Propiedades del cuaderno:

1. **Semillas fijas** para todo el pipeline.
2. **ROI = recorte por caja cuadrada con margen, sin enmascarar el fondo.**
3. Cada imagen registra su **estado `ROI_OK` / `ROI_DEGRADADA`**, sus cajas y el
   **tiempo de SAM** en un CSV congelado por categoría; toda re-ejecución parte
   de ese CSV sin volver a segmentar.
4. **Sin `pickle`**: los datos crudos de PatchCore se guardan en `.npz` y los
   bancos se exportan con manifiesto SHA-256.
5. **Re-proyección del mapa de calor a coordenadas de la imagen original** y
   AUROC píxel comparable entre línea base y ROI.

Secciones: entorno e instalación (1), dataset (2), pipeline y regla de
selección (3–4), corridas de PatchCore (5–7), resultados y figuras (8–9),
re-proyección (10), exportación de artefactos (11), guardado de resultados
(12) y galería de la aplicación (13).


## Verificación del entorno

In [ ]:
import sys, torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA version:', torch.version.cuda)
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Reproducibilidad: semillas fijas para todo el pipeline
import os, random
import numpy as np
import torch

SEMILLA = 0
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)
torch.cuda.manual_seed_all(SEMILLA)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ['PYTHONHASHSEED'] = str(SEMILLA)
print('Semilla fija:', SEMILLA)


## 1. Instalación y repositorios

Se instalan:
- **PatchCore** (repositorio Amazon Science)  
- **SAM** v1 (`segment-anything`, Meta AI) — compatible con Colab estándar


In [ ]:
# PatchCore
!git clone --quiet https://github.com/amazon-science/patchcore-inspection.git
!pip -q install -r patchcore-inspection/requirements.txt
!pip -q install -e patchcore-inspection

In [ ]:

# SAM 1 — pip install directo desde el repo oficial
!pip -q install git+https://github.com/facebookresearch/segment-anything.git

print('Instalación completada')


## 2. Descarga del dataset MVTec AD

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp /content/drive/MyDrive/Kaggle/kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json
print('Credenciales de Kaggle configuradas')


In [ ]:
!pip -q install kaggle
!kaggle datasets download -d ipythonx/mvtec-ad -p /content/data --quiet
!unzip -q /content/data/mvtec-ad.zip -d /content/data
!ls /content/data | head -15


In [ ]:
from pathlib import Path

DATA_ROOT = Path('/content/data')

# Corrida completa de UNA categoría: cambia solo CATEGORY y usa
# "Entorno de ejecución → Ejecutar todo". La misma variable gobierna la
# segmentación, las corridas de PatchCore, la exportación de artefactos y
# la galería de la aplicación (sección 13). Requisitos previos de la sesión:
# kaggle.json subido a /content (celda de descarga) y autorizar Drive.
CATEGORY  = 'cable'

CATEGORIAS_A_EXPORTAR = [CATEGORY]     # artefactos que se exportan en la sección 11
PERCENTIL_UMBRAL      = 99             # percentil del umbral sobre normales de validación
FRACCION_VALIDACION   = 0.15           # normales de train reservadas para calibrar el umbral

# Detectar MVTEC_ROOT
def find_mvtec_root(base_dir, category):
    base_dir = Path(base_dir)
    if (base_dir / category).exists():
        return base_dir
    for sub in base_dir.iterdir():
        if sub.is_dir() and (sub / category).exists():
            return sub
    return None

MVTEC_ROOT = find_mvtec_root(DATA_ROOT, CATEGORY)
if MVTEC_ROOT is None:
    raise FileNotFoundError(f'No se encontró la carpeta {CATEGORY} dentro de {DATA_ROOT}')
print('MVTec root:', MVTEC_ROOT)
print('Categoría:', CATEGORY)
print('Train/good existe:', (MVTEC_ROOT / CATEGORY / 'train' / 'good').exists())


## 2.1. Visualización de muestras del dataset

Se muestran ejemplos representativos de MVTec AD — categoría **capsule** —
para ilustrar la variabilidad de imágenes normales y los tipos de defectos presentes.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import random, os

def show_dataset_samples(mvtec_root, category, n_normal=4, n_defect_per_type=2, seed=42):
    random.seed(seed)
    cat_root = mvtec_root / category

    # Imágenes normales (train/good)
    normal_dir = cat_root / 'train' / 'good'
    normal_imgs = sorted(normal_dir.glob('*.png'))
    selected_normal = random.sample(normal_imgs, min(n_normal, len(normal_imgs)))

    # Tipos de defecto (test/<tipo> excepto 'good')
    test_root = cat_root / 'test'
    defect_types = [d for d in sorted(os.listdir(test_root)) if d != 'good']

    fig_rows = 1 + len(defect_types)
    fig, axes = plt.subplots(fig_rows, max(n_normal, n_defect_per_type + 1),
                             figsize=(14, 3.5 * fig_rows))
    if fig_rows == 1:
        axes = [axes]

    # Fila 0: normales
    for col, p in enumerate(selected_normal):
        axes[0][col].imshow(Image.open(p))
        axes[0][col].set_title('Normal', fontsize=9)
        axes[0][col].axis('off')
    for col in range(len(selected_normal), axes[0].shape[0] if hasattr(axes[0], '__len__') else n_normal):
        try: axes[0][col].axis('off')
        except: pass

    # Filas siguientes: cada tipo de defecto + su máscara GT
    for row, dtype in enumerate(defect_types, start=1):
        defect_imgs = sorted((test_root / dtype).glob('*.png'))
        selected_defect = random.sample(defect_imgs, min(n_defect_per_type, len(defect_imgs)))
        col = 0
        for p in selected_defect:
            axes[row][col].imshow(Image.open(p))
            axes[row][col].set_title(f'{dtype}', fontsize=9)
            axes[row][col].axis('off')
            col += 1
            # Máscara GT correspondiente
            gt_path = cat_root / 'ground_truth' / dtype / (p.stem + '_mask.png')
            if gt_path.exists():
                axes[row][col].imshow(Image.open(gt_path), cmap='Reds', vmin=0, vmax=255)
                axes[row][col].set_title('GT mask', fontsize=9)
                axes[row][col].axis('off')
                col += 1
        for c in range(col, len(axes[row]) if hasattr(axes[row], '__len__') else n_normal):
            try: axes[row][c].axis('off')
            except: pass

    plt.suptitle(f'MVTec AD — Categoría: {category}', fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('/content/dataset_samples.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Figura guardada en /content/dataset_samples.png')

show_dataset_samples(MVTEC_ROOT, CATEGORY)


## 3. Pipeline: SAM 1 → PatchCore

### Arquitectura propuesta

```
Imagen original (MVTec AD)
        │
        ▼
  ┌─────────────┐
  │    SAM 1    │  ← SamAutomaticMaskGenerator (sin ajuste fino ni prompts)
  │  ViT-H      │     Genera todas las máscaras candidatas
  │             │     Regla de selección por puntuación → bounding box
  └─────┬───────┘
        │  ROI = recorte por bbox SIN enmascarar el fondo
        ▼
  ┌─────────────────────┐
  │     PatchCore        │  ← Backbone: WideResNet-50-2 (ImageNet)
  │  (Banco de memoria   │     Capas: layer2 + layer3
  │   de parches         │     Coreset: 10 % approx greedy
  │   normales)          │     Fase de preparación (sin ajuste de parámetros)
  └─────────────────────┘
        │
        ▼
  Anomaly score (imagen) + Heatmap (píxel, re-proyectado al final)
```

El **Baseline** omite el bloque SAM y pasa la imagen completa directamente a PatchCore.


In [ ]:
# Descarga del checkpoint SAM 1 — ViT-H (el del paper original)
import hashlib

SAM1_CKPT = DATA_ROOT / 'sam_vit_h_4b8939.pth'

if not SAM1_CKPT.exists():
    !wget -q -O {SAM1_CKPT} \
        https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
    print('Checkpoint descargado:', SAM1_CKPT)
else:
    print('Checkpoint ya existe:', SAM1_CKPT)

# SHA-256 del checkpoint: queda registrado para verificarlo en el despliegue
_h = hashlib.sha256()
with open(SAM1_CKPT, 'rb') as _f:
    for _bloque in iter(lambda: _f.read(1 << 22), b''):
        _h.update(_bloque)
print('SHA-256 del checkpoint:', _h.hexdigest())


In [ ]:
import numpy as np
from PIL import Image
from tqdm import tqdm

from segment_anything import SamAutomaticMaskGenerator, sam_model_registry

device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'

sam = sam_model_registry['vit_h'](checkpoint=str(SAM1_CKPT))
sam.to(device)
mask_generator = SamAutomaticMaskGenerator(sam)
print(f'SAM 1 (vit_h) listo en {device}')


## 4.0. Regla de selección de máscara y validación previa

La regla de selección (parámetros fijados a priori): filtro de área
**3 %–95 %** de la imagen, objetivo de proporción **ρ = 0.35**, y puntuación
`0.60·IoU + 0.40·estabilidad + 0.25·relleno − 0.15·contacto_borde − |ratio − ρ|`.

- Si alguna máscara pasa los filtros → **`ROI_OK`** y se usa la mejor puntuada.
- Si ninguna pasa → **`ROI_DEGRADADA`**: se usa la máscara de mayor área o, en el
  peor caso, la imagen completa. El estado queda registrado por imagen.

La ROI definitiva es el **recorte por bounding box sin enmascarar el fondo**.


In [ ]:
from pathlib import Path
from matplotlib.patches import Rectangle
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Parámetros de la regla de selección (fijados a priori)
AREA_MIN_RATIO = 0.03
AREA_MAX_RATIO = 0.95
RHO_OBJETIVO   = 0.35
PESO_IOU, PESO_ESTABILIDAD, PESO_RELLENO, PESO_BORDE = 0.60, 0.40, 0.25, 0.15

def _mask_bbox(mask):
    ys, xs = np.where(mask)
    if len(xs) == 0 or len(ys) == 0:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())

def _score_mask(mask_dict, image_size):
    w, h = image_size
    seg = mask_dict.get('segmentation', None)
    if seg is None:
        return None
    area = float(mask_dict.get('area', float(np.sum(seg))))
    ratio = area / float(w * h)
    if ratio < AREA_MIN_RATIO or ratio > AREA_MAX_RATIO:
        return None
    bbox = _mask_bbox(seg)
    if bbox is None:
        return None
    x0, y0, x1, y1 = bbox
    bbox_area = max((x1 - x0 + 1) * (y1 - y0 + 1), 1)
    fill = area / bbox_area
    edge_touch = int(x0 <= 1) + int(y0 <= 1) + int(x1 >= w - 2) + int(y1 >= h - 2)
    score = (
        PESO_IOU * float(mask_dict.get('predicted_iou', 0.0)) +
        PESO_ESTABILIDAD * float(mask_dict.get('stability_score', 0.0)) +
        PESO_RELLENO * fill -
        PESO_BORDE * edge_touch -
        abs(ratio - RHO_OBJETIVO)
    )
    return score, bbox, seg, mask_dict

def select_sam_foreground_mask(masks, image_size):
    """Devuelve (bbox, seg, meta, estado) con estado ROI_OK | ROI_DEGRADADA."""
    if not masks:
        return None, None, None, 'ROI_DEGRADADA'

    puntuadas = []
    for m in masks:
        item = _score_mask(m, image_size)
        if item is not None:
            puntuadas.append(item)

    if puntuadas:
        puntuadas.sort(key=lambda t: t[0], reverse=True)
        _, bbox, seg, meta = puntuadas[0]
        return bbox, seg, meta, 'ROI_OK'

    # Respaldo: ninguna máscara pasó los filtros → la de mayor área (degradada)
    mejor = max(masks, key=lambda m: float(m.get('area', 0)))
    seg = mejor.get('segmentation', None)
    bbox = _mask_bbox(seg) if seg is not None else None
    return bbox, seg, mejor, 'ROI_DEGRADADA'

def preview_sam_segmentation(image_path, title=None, save_path=None):
    image = Image.open(image_path).convert('RGB')
    masks = mask_generator.generate(np.array(image))
    bbox, seg, meta, estado = select_sam_foreground_mask(masks, image.size)

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    axes[0].imshow(image)
    axes[0].set_title('Original')
    axes[0].axis('off')

    axes[1].imshow(image)
    if seg is not None:
        axes[1].imshow(seg, cmap='jet', alpha=0.45)
    if bbox is not None:
        x0, y0, x1, y1 = bbox
        axes[1].add_patch(Rectangle((x0, y0), x1 - x0 + 1, y1 - y0 + 1,
                                    fill=False, linewidth=2))
    axes[1].set_title(f'Máscara seleccionada · {estado}')
    axes[1].axis('off')

    # La ROI definitiva es la caja CUADRADA con margen; la caja
    # cruda del selector se dibuja arriba solo como referencia. Si
    # BBOX_CUADRADO aun no esta definido (celda posterior), se muestra la cruda.
    bbox_final = bbox
    if bbox is not None and globals().get('BBOX_CUADRADO', False):
        bbox_final = _expandir_bbox(bbox, image.size)
        x0, y0, x1, y1 = bbox_final
        axes[1].add_patch(Rectangle((x0, y0), x1 - x0 + 1, y1 - y0 + 1,
                                    fill=False, linewidth=2, linestyle='--'))
    if bbox_final is not None:
        crop = image.crop((bbox_final[0], bbox_final[1], bbox_final[2] + 1, bbox_final[3] + 1))
        axes[2].imshow(crop)
        titulo_roi = ('ROI: caja cuadrada con margen'
                      if globals().get('BBOX_CUADRADO', False)
                      else 'ROI: recorte por bbox (sin enmascarar)')
        axes[2].set_title(titulo_roi)
    else:
        axes[2].imshow(image)
        axes[2].set_title('Respaldo: imagen completa')
    axes[2].axis('off')

    if title:
        plt.suptitle(title, fontsize=12, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=140, bbox_inches='tight')
        print('Figura guardada en', save_path)
    plt.show()

if 'mask_generator' not in globals():
    print('SAM no está cargado: se omite la validación visual.')
else:
    good_sample = sorted((MVTEC_ROOT / CATEGORY / 'test' / 'good').glob('*.png'))[:1]
    if good_sample:
        preview_sam_segmentation(
            good_sample[0],
            title=f'Validación SAM 1 — {CATEGORY} / test / good',
            save_path='/content/sam_validation_good.png'
        )

    defect_dirs = [d for d in sorted((MVTEC_ROOT / CATEGORY / 'test').iterdir()) if d.is_dir() and d.name != 'good']
    if defect_dirs:
        first_defect = sorted(defect_dirs[0].glob('*.png'))[:1]
        if first_defect:
            preview_sam_segmentation(
                first_defect[0],
                title=f'Validación SAM 1 — {CATEGORY} / test / {defect_dirs[0].name}',
                save_path='/content/sam_validation_defect.png'
            )


### 4.0.1. Compuerta de consistencia (correr ANTES de generar la ROI)

Con ~90 s se verifica que la regla elige la **misma región** en imágenes
normales de la categoría. Si la desviación de área es alta o los centros
saltan (caso `cable`: 7/8 sub-partes en posiciones distintas),
**no** generar la ROI: llevar el caso a la decisión de la regla de selección
en vez de gastar la hora de SAM.


In [ ]:
# Compuerta: consistencia de la selección en normales (~90 s)
rutas_chequeo = sorted((MVTEC_ROOT / CATEGORY / 'train' / 'good').glob('*.png'))[:8]
areas_chk, centros_chk = [], []
for ruta_chk in rutas_chequeo:
    imagen_chk = Image.open(ruta_chk).convert('RGB')
    mascaras_chk = mask_generator.generate(np.array(imagen_chk))
    bbox_chk, _seg, _meta, estado_chk = select_sam_foreground_mask(mascaras_chk, imagen_chk.size)
    w_chk, h_chk = imagen_chk.size
    if bbox_chk is None:
        print(f'{ruta_chk.name}: {estado_chk} · sin bbox'); continue
    x0, y0, x1, y1 = bbox_chk
    area_chk = (x1 - x0 + 1) * (y1 - y0 + 1) / (w_chk * h_chk)
    cx, cy = (x0 + x1) / 2 / w_chk, (y0 + y1) / 2 / h_chk
    areas_chk.append(area_chk); centros_chk.append((cx, cy))
    print(f'{ruta_chk.name}: {estado_chk} · área bbox {area_chk:.2f} · centro ({cx:.2f}, {cy:.2f})')
if areas_chk:
    import numpy as _np
    dispersion_centros = float(_np.std(_np.array(centros_chk), axis=0).max())
    print(f'Área media {_np.mean(areas_chk):.2f} · desviación {_np.std(areas_chk):.2f} · dispersión de centros {dispersion_centros:.2f}')
    if _np.std(areas_chk) > 0.10 or dispersion_centros > 0.08:
        print('AVISO: selección INCONSISTENTE con la regla fijada; la categoría se evalúa igual y el resultado se reporta como limitación de la regla.')
    else:
        print('OK: selección consistente.')


## 4. Generación de ROI con SAM 1

Recorre train y test y recorta cada imagen a su caja (**sin enmascarar el
fondo**). Por imagen se registran el estado (`ROI_OK`/`ROI_DEGRADADA`), la
caja cruda de la máscara elegida, la caja cuadrada definitiva y el tiempo de
segmentación en `roi_estado_<categoría>.csv`. Ese registro congelado es el
punto de partida de toda re-ejecución: la subsección de reconstrucción, más
abajo, vuelve a generar los recortes desde él sin ejecutar SAM.


In [ ]:
import os, shutil, time
import pandas as pd

ROI_ROOT = DATA_ROOT / 'mvtec_sam_roi'

# Con cajas no cuadradas, Resize(256)+CenterCrop(224) descarta hasta el 75 %
# de la ROI (capsule: aspecto medio 3.12 → PatchCore vería solo ~25 %). La
# caja se expande a cuadrado con margen 256/224 para que el recorte central
# aterrice exactamente sobre la ROI, con fondo real.
BBOX_CUADRADO = True
MARGEN_CENTERCROP = 256 / 224

def _expandir_bbox(bbox, image_size, margen=MARGEN_CENTERCROP):
    w, h = image_size
    x0, y0, x1, y1 = bbox
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    lado = int(round(max(x1 - x0 + 1, y1 - y0 + 1) * margen))
    lado = min(lado, w, h)
    x0n = max(0, min(int(round(cx - lado / 2)), w - lado))
    y0n = max(0, min(int(round(cy - lado / 2)), h - lado))
    return (x0n, y0n, x0n + lado - 1, y0n + lado - 1)

def _safe_crop(image, bbox):
    x0, y0, x1, y1 = bbox
    return image.crop((x0, y0, x1 + 1, y1 + 1))

def _full_bbox(image):
    w, h = image.size
    return (0, 0, w - 1, h - 1)

def _ensure_mask(path, size):
    if path.exists():
        return
    Image.new('L', size, color=0).save(path)

def _mask_to_image_name(mask_name):
    stem = mask_name.rsplit('.', 1)[0]
    if stem.endswith('_mask'):
        stem = stem[:-5]
    return f"{stem}.png"

def build_roi_dataset(categoria=CATEGORY):
    src_root = MVTEC_ROOT / categoria
    dst_root = ROI_ROOT / categoria
    bbox_map = {}
    registro = []

    if dst_root.exists():
        shutil.rmtree(dst_root)
    dst_root.mkdir(parents=True, exist_ok=True)

    for split in ['train', 'test']:
        split_root = src_root / split
        bbox_map[split] = {}
        for defect_type in sorted(os.listdir(split_root)):
            defect_root = split_root / defect_type
            if not defect_root.is_dir():
                continue
            dst_defect_root = dst_root / split / defect_type
            dst_defect_root.mkdir(parents=True, exist_ok=True)
            bbox_map[split][defect_type] = {}

            for img_name in tqdm(sorted(os.listdir(defect_root)),
                                 desc=f'SAM {categoria} {split}/{defect_type}'):
                if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    continue
                img_path = defect_root / img_name
                image = Image.open(img_path).convert('RGB')

                t0 = time.perf_counter()
                masks = mask_generator.generate(np.array(image))
                bbox, seg, meta, estado = select_sam_foreground_mask(masks, image.size)
                t_sam_ms = (time.perf_counter() - t0) * 1000

                if bbox is None:
                    bbox = _full_bbox(image)   # respaldo final: imagen completa

                bbox_sam = bbox
                if BBOX_CUADRADO:
                    bbox = _expandir_bbox(bbox_sam, image.size)

                bbox_map[split][defect_type][img_name] = bbox
                registro.append({
                    'categoria': categoria, 'split': split, 'tipo': defect_type,
                    'imagen': img_name, 'estado': estado,
                    'x0': bbox[0], 'y0': bbox[1], 'x1': bbox[2], 'y1': bbox[3],
                    'sam_x0': bbox_sam[0], 'sam_y0': bbox_sam[1],
                    'sam_x1': bbox_sam[2], 'sam_y1': bbox_sam[3],
                    't_sam_ms': round(t_sam_ms, 1),
                })

                # ROI definitiva: recorte por bbox SIN enmascarar el fondo
                _safe_crop(image, bbox).save(dst_defect_root / img_name)

    # Ground truth (test únicamente), recortado con el mismo bbox
    gt_root = src_root / 'ground_truth'
    if gt_root.exists():
        for defect_type in sorted(os.listdir(gt_root)):
            defect_root = gt_root / defect_type
            if not defect_root.is_dir():
                continue
            dst_defect_root = dst_root / 'ground_truth' / defect_type
            dst_defect_root.mkdir(parents=True, exist_ok=True)
            for mask_name in tqdm(sorted(os.listdir(defect_root)),
                                  desc=f'GT {categoria} {defect_type}'):
                if not mask_name.lower().endswith('.png'):
                    continue
                img_name = _mask_to_image_name(mask_name)
                img_path = src_root / 'test' / defect_type / img_name
                if not img_path.exists():
                    Image.open(gt_root / defect_type / mask_name).save(
                        dst_defect_root / mask_name)
                    continue
                image = Image.open(img_path).convert('RGB')
                bbox = bbox_map.get('test', {}).get(defect_type, {}).get(
                    img_name, _full_bbox(image))
                gt_mask = Image.open(gt_root / defect_type / mask_name)
                _safe_crop(gt_mask, bbox).save(dst_defect_root / mask_name)

    # Asegura una máscara por imagen anómala
    test_root = dst_root / 'test'
    gt_root_out = dst_root / 'ground_truth'
    for defect_type in sorted(os.listdir(test_root)):
        if defect_type == 'good':
            continue
        gt_defect = gt_root_out / defect_type
        gt_defect.mkdir(parents=True, exist_ok=True)
        for img_path in sorted((test_root / defect_type).iterdir()):
            if img_path.suffix.lower() != '.png':
                continue
            mask_path = gt_defect / (img_path.stem + '_mask.png')
            _ensure_mask(mask_path, Image.open(img_path).size)

    df = pd.DataFrame(registro)
    ruta_csv = DATA_ROOT / f'roi_estado_{categoria}.csv'
    df.to_csv(ruta_csv, index=False)

    degradadas = df[df.estado == 'ROI_DEGRADADA']
    print(f'ROI dataset listo: {dst_root}')
    print(f'Registro: {ruta_csv}')
    print(f'Imágenes: {len(df)} · ROI_DEGRADADA: {len(degradadas)} '
          f'({100 * len(degradadas) / max(len(df), 1):.1f} %)')
    print(f'Tiempo SAM por imagen — media: {df.t_sam_ms.mean():.0f} ms · '
          f'p95: {df.t_sam_ms.quantile(0.95):.0f} ms')
    return df


In [ ]:
# Segmentación con SAM: una sola vez por categoría (alrededor de 1 h en una T4).
# Si ya existe el registro roi_estado_<categoría>.csv (carpeta resultados/ del
# repositorio), esta celda no es necesaria: la subsección de reconstrucción
# vuelve a generar los recortes desde el registro sin ejecutar SAM.
registro_roi = build_roi_dataset(CATEGORY)


In [ ]:
def check_roi_consistency(categoria=CATEGORY):
    test_root = ROI_ROOT / categoria / 'test'
    gt_root   = ROI_ROOT / categoria / 'ground_truth'
    print(f'\n--- Consistencia ROI ({categoria}) ---')
    for defect_type in sorted(os.listdir(test_root)):
        if defect_type == 'good':
            continue
        tc = len([p for p in (test_root / defect_type).iterdir()
                  if p.suffix.lower() == '.png'])
        gc = len([p for p in (gt_root / defect_type).iterdir()
                  if p.suffix.lower() == '.png']) if (gt_root / defect_type).exists() else 0
        status = 'ALINEADO' if tc == gc else 'DESALINEADO'
        print(f'  {status} {defect_type}: test={tc} gt={gc}')

check_roi_consistency(CATEGORY)


### Reconstrucción de la ROI desde el registro congelado

Para reproducir una categoría sin ejecutar SAM:

1. Ejecutar las celdas de entorno, semillas, instalación de PatchCore,
   dataset y configuración (`CATEGORY` y `CATEGORIAS_A_EXPORTAR` con la
   categoría). Las celdas del checkpoint de SAM, de carga del modelo y de
   generación con SAM no son necesarias.
2. Ejecutar la celda de la regla de selección y la de definiciones de ROI
   (solo definen funciones).
3. Copiar `roi_estado_<categoría>.csv` (carpeta `resultados/<categoría>/`
   del repositorio) a `/content/data/`.
4. Ejecutar `reconstruir_roi_desde_csv(CATEGORY)`: reconstruye los recortes
   y las máscaras de verdad de terreno con las cajas congeladas, en minutos.
5. Continuar con el flujo normal: parche npz, corridas de línea base y ROI,
   re-proyección, exportación de artefactos y guardado de resultados.

El registro conserva las cajas crudas de la máscara elegida (`sam_*`) y las
cajas cuadradas definitivas (`x0` a `y1`); con `BBOX_CUADRADO = False` se
reconstruye la variante de caja ajustada.


In [ ]:
# Reconstrucción de la ROI desde el registro congelado
# (roi_estado_<categoría>.csv) sin ejecutar SAM: es la vía de reproducción
# de las corridas.
import shutil
import pandas as pd
from tqdm import tqdm

def reconstruir_roi_desde_csv(categoria):
    ruta_csv = DATA_ROOT / f'roi_estado_{categoria}.csv'
    df = pd.read_csv(ruta_csv)
    usa_sam = 'sam_x0' in df.columns
    src_root = MVTEC_ROOT / categoria
    dst_root = ROI_ROOT / categoria
    if dst_root.exists():
        shutil.rmtree(dst_root)
    registro = []
    for _, fila in tqdm(df.iterrows(), total=len(df), desc=f'Reconstruyendo {categoria}'):
        split, tipo, nombre = fila['split'], fila['tipo'], fila['imagen']
        img_path = src_root / split / tipo / nombre
        image = Image.open(img_path).convert('RGB')
        cols = ['sam_x0', 'sam_y0', 'sam_x1', 'sam_y1'] if usa_sam else ['x0', 'y0', 'x1', 'y1']
        bbox_sam = tuple(int(fila[c]) for c in cols)
        bbox = _expandir_bbox(bbox_sam, image.size) if BBOX_CUADRADO else bbox_sam
        destino = dst_root / split / tipo
        destino.mkdir(parents=True, exist_ok=True)
        _safe_crop(image, bbox).save(destino / nombre)
        if split == 'test' and tipo != 'good':
            gt_origen = src_root / 'ground_truth' / tipo / (Path(nombre).stem + '_mask.png')
            if gt_origen.exists():
                gt_destino = dst_root / 'ground_truth' / tipo
                gt_destino.mkdir(parents=True, exist_ok=True)
                _safe_crop(Image.open(gt_origen), bbox).save(gt_destino / gt_origen.name)
        registro.append({
            'categoria': categoria, 'split': split, 'tipo': tipo, 'imagen': nombre,
            'estado': fila['estado'],
            'x0': bbox[0], 'y0': bbox[1], 'x1': bbox[2], 'y1': bbox[3],
            'sam_x0': bbox_sam[0], 'sam_y0': bbox_sam[1],
            'sam_x1': bbox_sam[2], 'sam_y1': bbox_sam[3],
            't_sam_ms': fila.get('t_sam_ms', 0.0) if hasattr(fila, 'get') else fila['t_sam_ms'],
        })
    # máscaras vacías para anómalas sin GT
    test_root = dst_root / 'test'
    for tipo_dir in sorted(test_root.iterdir()):
        if not tipo_dir.is_dir() or tipo_dir.name == 'good':
            continue
        gt_defect = dst_root / 'ground_truth' / tipo_dir.name
        gt_defect.mkdir(parents=True, exist_ok=True)
        for img_path in sorted(tipo_dir.glob('*.png')):
            mask_path = gt_defect / (img_path.stem + '_mask.png')
            _ensure_mask(mask_path, Image.open(img_path).size)
    pd.DataFrame(registro).to_csv(ruta_csv, index=False)
    print(f'ROI reconstruida: {dst_root} · CSV actualizado: {ruta_csv}')

# Ejemplo: reconstruir_roi_desde_csv('capsule')


## 4.1. Visualización: imagen original vs ROI segmentada por SAM 1

Se comparan ejemplos del conjunto de test para mostrar el recorte generado por SAM.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

def show_sam_crops(mvtec_root, roi_root, category, n=4, seed=0):
    random.seed(seed)
    test_good = mvtec_root / category / 'test' / 'good'
    roi_good  = roi_root   / category / 'test' / 'good'
    all_samples = sorted(list(test_good.glob('*.png')))
    samples = random.sample(all_samples, min(n, len(all_samples)))

    fig, axes = plt.subplots(2, len(samples), figsize=(3.5 * len(samples), 7))
    if len(samples) == 1:
        axes = np.array(axes).reshape(2, 1)

    for col, p in enumerate(samples):
        axes[0][col].imshow(Image.open(p))
        axes[0][col].set_title('Original', fontsize=9)
        axes[0][col].axis('off')

        roi_path = roi_good / p.name
        axes[1][col].imshow(Image.open(roi_path))
        axes[1][col].set_title('ROI — SAM 1', fontsize=9)
        axes[1][col].axis('off')

    plt.suptitle(f'Comparación Original vs ROI — SAM 1 — {category}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/content/sam_roi_comparison.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Figura guardada en /content/sam_roi_comparison.png')

show_sam_crops(MVTEC_ROOT, ROI_ROOT, CATEGORY)

Parche a `run_patchcore.py` para guardar los datos crudos (mapas, scores,
etiquetas y rutas) en **`.npz` — sin `pickle`**. Idempotente: limpia
cualquier inserción previa antes de aplicar.


In [ ]:
import os, re
from pathlib import Path

patchcore_script = Path("patchcore-inspection/bin/run_patchcore.py")
content = patchcore_script.read_text()

# 1. Corregir transform_std/mean (ImageNet stats)
old_new_pairs = [
    ("dataloaders[\"testing\"].dataset.transform_std",  "[0.229, 0.224, 0.225]"),
    ("dataloaders[\"testing\"].dataset.transform_mean", "[0.485, 0.456, 0.406]"),
]
for s_old, s_new in old_new_pairs:
    content = content.replace(s_old, s_new)

# 2. Limpiar inserciones previas (pickle o npz) y aplicar el guardado en npz
content = re.sub(
    r"# === GUARDAR DATOS CRUDOS.*?patchcore.utils.plot_segmentation_images\(",
    "patchcore.utils.plot_segmentation_images(",
    content, flags=re.DOTALL,
)

final_code = (
    "# === GUARDAR DATOS CRUDOS (npz, sin pickle) ===\n"
    "                import numpy as _np\n"
    "                _dd = os.path.join(run_save_path, \"anomaly_data\")\n"
    "                os.makedirs(_dd, exist_ok=True)\n"
    "                _np.savez_compressed(\n"
    "                    os.path.join(_dd, \"anomaly_data.npz\"),\n"
    "                    segmentations=_np.asarray(segmentations, dtype=_np.float32),\n"
    "                    scores=_np.asarray(scores, dtype=_np.float32),\n"
    "                    labels=_np.asarray(anomaly_labels),\n"
    "                    image_paths=_np.asarray([str(_p) for _p in image_paths]),\n"
    "                )\n"
    "                patchcore.utils.plot_segmentation_images("
)

if "patchcore.utils.plot_segmentation_images(" in content:
    content = content.replace("patchcore.utils.plot_segmentation_images(", final_code)
    print("Parche de guardado (npz) aplicado correctamente.")
else:
    print("No se encontró el punto de inserción.")

patchcore_script.write_text(content)


## 5. PatchCore — Baseline (imagen completa)

PatchCore se **prepara** y evalúa directamente sobre las imágenes originales de MVTec AD.
**Backbone**: WideResNet-50-2 (ImageNet) — capas `layer2` + `layer3`, coreset 10 %.


In [ ]:
BASELINE_OUT = DATA_ROOT / f'results_baseline_{CATEGORY}'
BASELINE_OUT.mkdir(parents=True, exist_ok=True)

cmd = ' '.join([
    'env', 'PYTHONPATH=patchcore-inspection/src', 'python',
    'patchcore-inspection/bin/run_patchcore.py',
    '--gpu', '0', '--seed', '0', '--save_patchcore_model', '--save_segmentation_images',
    '--log_group', 'IM224_WR50_L2-3_P01_D1024-1024_PS-3_AN-1_S0',
    str(BASELINE_OUT),
    'patch_core',
    '-b', 'wideresnet50', '-le', 'layer2', '-le', 'layer3',
    '--pretrain_embed_dimension', '1024', '--target_embed_dimension', '1024',
    '--anomaly_scorer_num_nn', '1', '--patchsize', '3',
    'sampler', '-p', '0.1', 'approx_greedy_coreset',
    'dataset', '--resize', '256', '--imagesize', '224',
    '-d', CATEGORY, 'mvtec', str(MVTEC_ROOT)
])

print('Ejecutando baseline...')
!{cmd}


## 6. PatchCore — SAM 1 + ROI

In [ ]:
ROI_OUT = DATA_ROOT / f'results_sam_roi_{CATEGORY}'
ROI_OUT.mkdir(parents=True, exist_ok=True)

cmd = ' '.join([
    'env', 'PYTHONPATH=patchcore-inspection/src', 'python',
    'patchcore-inspection/bin/run_patchcore.py',
    '--gpu', '0', '--seed', '0', '--save_patchcore_model', '--save_segmentation_images',
    '--log_group', 'IM224_WR50_L2-3_P01_D1024-1024_PS-3_AN-1_S0_SAM1ROI',
    str(ROI_OUT),
    'patch_core',
    '-b', 'wideresnet50', '-le', 'layer2', '-le', 'layer3',
    '--pretrain_embed_dimension', '1024', '--target_embed_dimension', '1024',
    '--anomaly_scorer_num_nn', '1', '--patchsize', '3',
    'sampler', '-p', '0.1', 'approx_greedy_coreset',
    'dataset', '--resize', '256', '--imagesize', '224',
    '-d', CATEGORY, 'mvtec', str(ROI_ROOT)
])

print('Ejecutando SAM 1 + PatchCore...')
!{cmd}


In [ ]:
from pathlib import Path
import shutil

def _copy_results_csv(out_root: Path) -> Path | None:
    out_root = Path(out_root)
    direct = out_root / "results.csv"
    if direct.exists():
        return direct
    candidates = list(out_root.rglob("results.csv"))
    if not candidates:
        return None
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    src = candidates[0]
    shutil.copy2(src, direct)
    return direct

for name, out_dir in [("baseline", BASELINE_OUT), ("sam_roi", ROI_OUT)]:
    copied = _copy_results_csv(out_dir)
    if copied:
        print(f"results.csv ({name}) listo en: {copied}")
    else:
        print(f"No se encontro results.csv para {name}")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

def load_results(results_root):
    csv_path = Path(results_root) / 'results.csv'
    if not csv_path.exists():
        print('No se encontró:', csv_path)
        return None
    df = pd.read_csv(csv_path)
    if 'dataset' not in df.columns and 'Row Names' in df.columns:
        # Normaliza el nombre de la clase para que coincida con CATEGORY.
        df['dataset'] = (
            df['Row Names']
            .astype(str)
            .str.replace('mvtec_', '', regex=False)
            .str.replace('mvtec-', '', regex=False)
        )
    return df

baseline_df = load_results(BASELINE_OUT)
roi_df      = load_results(ROI_OUT)


def _load_anomaly_data(results_dir):
    """Carga anomaly_data.npz (sin pickle)."""
    candidates = list(Path(results_dir).rglob("anomaly_data.npz"))
    if not candidates:
        return None
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    datos = np.load(candidates[0], allow_pickle=False)
    return {k: datos[k] for k in datos.files}


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_comparison_table(baseline_df, roi_df, category, save_path='/content/results_table.png'):
    metrics  = ['AUROC imagen', 'AUROC píxel (full)', 'AUROC píxel (anomalía)']
    col_keys = ['instance_auroc', 'full_pixel_auroc', 'anomaly_pixel_auroc']

    def _get(df):
        if df is None:
            return [np.nan] * 3
        row = df[df['dataset'] == category]
        if row.empty:
            return [np.nan] * 3
        vals = []
        for k in col_keys:
            vals.append(float(row[k].values[0]) if k in row.columns else np.nan)
        return vals

    baseline_vals = _get(baseline_df)
    roi_vals      = _get(roi_df)

    x = np.arange(len(metrics))
    w = 0.35
    fig, ax = plt.subplots(figsize=(9, 5))
    bars1 = ax.bar(x - w / 2, np.nan_to_num(baseline_vals, nan=0.0), w, label='Baseline')
    bars2 = ax.bar(x + w / 2, np.nan_to_num(roi_vals, nan=0.0), w, label='SAM 1 + PatchCore')

    for bar, val in list(zip(bars1, baseline_vals)) + list(zip(bars2, roi_vals)):
        if np.isnan(val):
            continue
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.002,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

    all_vals = [v for v in (baseline_vals + roi_vals) if not np.isnan(v)]
    if all_vals:
        ymin = max(0.0, min(all_vals) - 0.05)
        ymax = min(1.01, max(all_vals) + 0.02)
        if ymax - ymin < 0.05:
            ymax = min(1.01, ymin + 0.05)
        ax.set_ylim(ymin, ymax)
    else:
        ax.set_ylim(0.0, 1.0)
    ax.set_ylabel('AUROC', fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(metrics, fontsize=10)
    ax.set_title(f'Comparación de métricas AUROC — {category} (MVTec AD)', fontsize=12)
    ax.legend(fontsize=10)
    ax.yaxis.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print('Figura guardada en', save_path)

plot_comparison_table(baseline_df, roi_df, CATEGORY,
                      save_path=f'/content/results_table_{CATEGORY}.png')

## 8. Resultados visuales: mapas de calor de anomalías

Se muestran, para imágenes con defectos de la clase *capsule*, la imagen original,  
la ROI recortada por SAM 1 y el mapa de calor generado por PatchCore.  
Esto permite evaluar cualitativamente si la segmentación mejora la localización.


In [ ]:
import os, random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

def _find_matching_index(data, image_name):
    for i, path in enumerate(data['image_paths']):
        if image_name in str(path):
            return i
    return None

def show_anomaly_heatmaps_v2(mvtec_root, roi_root, baseline_out, roi_out, category, n=3):
    b_data = _load_anomaly_data(baseline_out)
    r_data = _load_anomaly_data(roi_out)

    if b_data is None or r_data is None:
        print("No se encontraron los anomaly_data.npz: vuelve a ejecutar las celdas de PatchCore tras aplicar el parche.")
        return

    test_root = Path(mvtec_root) / category / "test"
    defects = [d for d in os.listdir(test_root) if d != "good"]

    for dtype in defects[:2]:
        imgs = sorted((test_root / dtype).glob("*.png"))
        selected = random.sample(imgs, min(n, len(imgs)))

        fig, axes = plt.subplots(len(selected), 5, figsize=(18, len(selected) * 3.5))
        if len(selected) == 1:
            axes = [axes]

        for row, p in enumerate(selected):
            name = p.name
            idx_b = _find_matching_index(b_data, name)
            idx_r = _find_matching_index(r_data, name)

            axes[row][0].imshow(Image.open(p))
            axes[row][0].set_title("Original")

            roi_p = Path(roi_root) / category / "test" / dtype / name
            if roi_p.exists():
                axes[row][1].imshow(Image.open(roi_p))
                axes[row][1].set_title("ROI (SAM)")

            if idx_b is not None:
                axes[row][2].imshow(b_data['segmentations'][idx_b], cmap='jet')
                axes[row][2].set_title(f"Base (Score: {b_data['scores'][idx_b]:.2f})")

            if idx_r is not None:
                axes[row][3].imshow(r_data['segmentations'][idx_r], cmap='jet')
                axes[row][3].set_title(f"ROI (Score: {r_data['scores'][idx_r]:.2f})")

            gt_p = Path(mvtec_root) / category / "ground_truth" / dtype / (p.stem + "_mask.png")
            if gt_p.exists():
                axes[row][4].imshow(Image.open(gt_p), cmap='gray')
                axes[row][4].set_title("Ground Truth")

            for ax in axes[row]:
                ax.axis('off')

        plt.tight_layout()
        plt.show()

show_anomaly_heatmaps_v2(MVTEC_ROOT, ROI_ROOT, BASELINE_OUT, ROI_OUT, CATEGORY)


## 9. Curvas ROC

Se grafican las curvas ROC a nivel de imagen y a nivel de píxel para ambas configuraciones,
mostrando la separabilidad entre imágenes/píxeles normales y anómalos.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

def plot_actual_roc_curves(baseline_out, roi_out, category):
    b_data = _load_anomaly_data(baseline_out)
    r_data = _load_anomaly_data(roi_out)

    if b_data is None or r_data is None:
        print("No se encontraron datos crudos para generar curvas reales.")
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, data, title, color in [
        (axes[0], b_data, "ROC: Baseline (Imagen Completa)", "#4C72B0"),
        (axes[1], r_data, "ROC: SAM 1 + PatchCore (ROI)", "#DD8452")
    ]:
        labels = np.array(data["labels"])
        scores = np.array(data["scores"])
        fpr, tpr, _ = roc_curve(labels, scores)
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, lw=2.5, label=f'AUC = {roc_auc:.4f}')
        ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
        ax.set_xlim([-0.01, 1.0])
        ax.set_ylim([0.0, 1.05])
        ax.set_xlabel('Tasa de Falsos Positivos (FPR)')
        ax.set_ylabel('Tasa de Verdaderos Positivos (TPR)')
        ax.set_title(title)
        ax.legend(loc="lower right")
        ax.grid(True, linestyle=':', alpha=0.6)

    plt.suptitle(f'Curvas ROC (Nivel Imagen) — Categoría: {category}', fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(f'/content/roc_curves_reales_{category}.png', dpi=150)
    plt.show()

plot_actual_roc_curves(BASELINE_OUT, ROI_OUT, CATEGORY)


## 10. Re-proyección del mapa y AUROC píxel comparable

El mapa de calor de la configuración ROI vive en coordenadas del recorte
(224×224). Para que el AUROC píxel sea comparable con el baseline, ambos se
evalúan **en coordenadas de la imagen original**: se invierte
`CenterCrop(224) ∘ Resize(256)` y, en el caso ROI, el mapa se coloca dentro
del bbox registrado en `roi_estado_<categoría>.csv`. La métrica primaria del
proyecto es este AUROC re-proyectado; el AUROC dentro-de-ROI queda como
secundaria.

> Nota: concatena todos los píxeles del test (~centenares de MB); en Colab
> estándar tarda un par de minutos.


In [ ]:
import cv2
import pandas as pd
from sklearn.metrics import roc_auc_score

def reproyectar_mapa(mapa, bbox, tam_original, resize=256, imagesize=224):
    """Invierte CenterCrop(imagesize) ∘ Resize(resize) y devuelve el mapa
    en coordenadas de la imagen original (fuera del bbox queda 0)."""
    H, W = tam_original
    x0, y0, x1, y1 = bbox
    bw, bh = x1 - x0 + 1, y1 - y0 + 1
    if bw <= bh:
        rw, rh = resize, int(round(resize * bh / bw))
    else:
        rh, rw = resize, int(round(resize * bw / bh))
    rw, rh = max(rw, imagesize), max(rh, imagesize)
    lienzo = np.zeros((rh, rw), dtype=np.float32)
    oy, ox = (rh - imagesize) // 2, (rw - imagesize) // 2
    lienzo[oy:oy + imagesize, ox:ox + imagesize] = mapa
    mapa_bbox = cv2.resize(lienzo, (bw, bh), interpolation=cv2.INTER_LINEAR)
    completo = np.zeros((H, W), dtype=np.float32)
    completo[y0:y1 + 1, x0:x1 + 1] = mapa_bbox
    return completo

def auroc_pixel_reproyectado(datos, categoria, usar_bbox):
    registro = None
    if usar_bbox:
        registro = pd.read_csv(DATA_ROOT / f'roi_estado_{categoria}.csv')
        registro = registro[registro.split == 'test']
    puntajes, etiquetas = [], []
    for i, ruta in enumerate(datos['image_paths']):
        ruta = Path(str(ruta))
        tipo, nombre = ruta.parent.name, ruta.name
        original = MVTEC_ROOT / categoria / 'test' / tipo / nombre
        if not original.exists():
            continue
        W, H = Image.open(original).size
        if usar_bbox:
            fila = registro[(registro.tipo == tipo) & (registro.imagen == nombre)]
            if fila.empty:
                continue
            bbox = tuple(int(v) for v in fila.iloc[0][['x0', 'y0', 'x1', 'y1']])
        else:
            bbox = (0, 0, W - 1, H - 1)
        mapa = reproyectar_mapa(datos['segmentations'][i].astype(np.float32), bbox, (H, W))
        if tipo == 'good':
            gt = np.zeros((H, W), dtype=bool)
        else:
            gt_ruta = MVTEC_ROOT / categoria / 'ground_truth' / tipo / (ruta.stem + '_mask.png')
            gt = np.array(Image.open(gt_ruta).convert('L')) > 0
            if gt.shape != (H, W):
                gt = cv2.resize(gt.astype(np.uint8), (W, H),
                                interpolation=cv2.INTER_NEAREST).astype(bool)
        puntajes.append(mapa.ravel().astype(np.float16))
        etiquetas.append(gt.ravel())
    y = np.concatenate(etiquetas)
    s = np.concatenate(puntajes).astype(np.float32)
    return roc_auc_score(y, s)

b_data = _load_anomaly_data(BASELINE_OUT)
r_data = _load_anomaly_data(ROI_OUT)
if b_data is not None and r_data is not None:
    auroc_base = auroc_pixel_reproyectado(b_data, CATEGORY, usar_bbox=False)
    auroc_roi  = auroc_pixel_reproyectado(r_data, CATEGORY, usar_bbox=True)
    print(f'AUROC píxel re-proyectado (coordenadas originales) — {CATEGORY}')
    print(f'  Baseline (imagen completa): {auroc_base:.4f}')
    print(f'  SAM 1 + ROI:                {auroc_roi:.4f}')
    pd.DataFrame([{'categoria': CATEGORY,
                   'auroc_pixel_reproyectado_baseline': round(auroc_base, 6),
                   'auroc_pixel_reproyectado_roi': round(auroc_roi, 6)}]).to_csv(
        DATA_ROOT / f'auroc_reproyectado_{CATEGORY}.csv', index=False)
    print('Guardado:', DATA_ROOT / f'auroc_reproyectado_{CATEGORY}.csv')
else:
    print('Faltan los anomaly_data.npz: ejecuta primero las corridas de PatchCore.')


## 11. Preparación y exportación de artefactos para la aplicación web

Para cada categoría de `CATEGORIAS_A_EXPORTAR` (empezando por `capsule`):

1. Se separa un **split de validación** de imágenes normales del train ROI
   (fracción `FRACCION_VALIDACION`, semilla fija) — el banco se prepara **sin**
   esas imágenes para que sus puntuaciones sirvan de referencia.
2. **Fase de preparación** de PatchCore con la misma configuración del
   experimento (WideResNet-50-2, layer2+layer3, 1024/1024, patchsize 3,
   coreset 10 % approx greedy, k=1).
3. El **umbral** es el percentil `PERCENTIL_UMBRAL` (p99)
   de las puntuaciones del split de validación.
4. Se exporta `banco.npz` (float32 `[N, D]`, `np.savez_compressed` — sin
   `pickle`), `calibracion.json` y `manifiesto.json` con el SHA-256 de
   cada archivo, y se **verifica** recargando con
   `allow_pickle=False` y recomputando los hashes.

La carpeta resultante (`artefactos/<categoría>/`) se copia tal cual a
`backend/artefactos/<categoría>/` de la aplicación.


In [ ]:
import json, time
import torchvision
import sys
from pathlib import Path

# El repo de PatchCore no es instalable con pip (sin setup.py):
# se importa directo desde su carpeta src, igual que hace su propio CLI.
RUTA_PATCHCORE = Path('patchcore-inspection/src').resolve()
if str(RUTA_PATCHCORE) not in sys.path:
    sys.path.insert(0, str(RUTA_PATCHCORE))

import patchcore.backbones
import patchcore.common
import patchcore.patchcore
import patchcore.sampler
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch

# 'device' se define en la celda de SAM; si esa celda no se ejecutó
# (reconstrucción desde el registro) se define aquí.
if 'device' not in globals():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

RESIZE, IMAGESIZE = 256, 224
MEDIA_IMAGENET = [0.485, 0.456, 0.406]
STD_IMAGENET   = [0.229, 0.224, 0.225]

_transformacion = transforms.Compose([
    transforms.Resize(RESIZE),
    transforms.CenterCrop(IMAGESIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEDIA_IMAGENET, std=STD_IMAGENET),
])

class DatasetListaImagenes(Dataset):
    """Mismas transformaciones que el dataset MVTec del repo de PatchCore."""
    def __init__(self, rutas):
        self.rutas = list(rutas)
    def __len__(self):
        return len(self.rutas)
    def __getitem__(self, i):
        return _transformacion(Image.open(self.rutas[i]).convert('RGB'))

def preparar_patchcore(rutas_preparacion):
    """Fase de preparación: construye el banco de memoria (coreset 10 %)."""
    backbone = patchcore.backbones.load('wideresnet50')
    backbone.name, backbone.seed = 'wideresnet50', SEMILLA
    instancia = patchcore.patchcore.PatchCore(device)
    instancia.load(
        backbone=backbone,
        layers_to_extract_from=['layer2', 'layer3'],
        device=device,
        input_shape=(3, IMAGESIZE, IMAGESIZE),
        pretrain_embed_dimension=1024,
        target_embed_dimension=1024,
        patchsize=3,
        featuresampler=patchcore.sampler.ApproximateGreedyCoresetSampler(
            percentage=0.10, device=device),
        # la firma real de load() es anomaly_score_num_nn (sin 'r');
        # el CLI del repo pasa 'anomaly_scorer_num_nn' y cae en **kwargs.
        anomaly_score_num_nn=1,
        nn_method=patchcore.common.FaissNN(False, 4),
    )
    loader = DataLoader(DatasetListaImagenes(rutas_preparacion),
                        batch_size=8, shuffle=False, num_workers=2)
    instancia.fit(loader)
    banco = getattr(instancia.anomaly_scorer, 'detection_features', None)
    if banco is None:
        raise RuntimeError('Esta versión del repo de PatchCore no expone '
                           'anomaly_scorer.detection_features; revisar src/patchcore/common.py')
    banco = np.asarray(banco, dtype=np.float32)
    if banco.ndim != 2:
        raise RuntimeError(f'Banco con forma inesperada: {banco.shape}')
    return instancia, banco

def puntuar(instancia, rutas):
    """Puntuación imagen a imagen, con tiempo de detección por imagen."""
    puntuaciones, tiempos_ms = [], []
    loader = DataLoader(DatasetListaImagenes(rutas),
                        batch_size=1, shuffle=False, num_workers=2)
    for lote in loader:
        t0 = time.perf_counter()
        scores, _mapas = instancia._predict(lote)
        tiempos_ms.append((time.perf_counter() - t0) * 1000)
        puntuaciones.extend(float(v) for v in scores)
    return np.array(puntuaciones), np.array(tiempos_ms)


In [ ]:
import hashlib

ARTEFACTOS_ROOT = DATA_ROOT / 'artefactos'

def _sha256_archivo(ruta):
    h = hashlib.sha256()
    with open(ruta, 'rb') as f:
        for bloque in iter(lambda: f.read(1 << 20), b''):
            h.update(bloque)
    return h.hexdigest()

def exportar_artefactos(categoria, banco, calibracion):
    carpeta = ARTEFACTOS_ROOT / categoria
    carpeta.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(carpeta / 'banco.npz', banco=banco)
    (carpeta / 'calibracion.json').write_text(
        json.dumps(calibracion, indent=2, ensure_ascii=False), encoding='utf-8')
    manifiesto = {
        'categoria': categoria,
        'version': '1',
        'archivos': {
            'banco.npz': _sha256_archivo(carpeta / 'banco.npz'),
            'calibracion.json': _sha256_archivo(carpeta / 'calibracion.json'),
        },
    }
    (carpeta / 'manifiesto.json').write_text(
        json.dumps(manifiesto, indent=2, ensure_ascii=False), encoding='utf-8')
    return carpeta

def verificar_artefactos(carpeta):
    """Relee y valida: hashes del manifiesto + carga sin pickle."""
    manifiesto = json.loads((carpeta / 'manifiesto.json').read_text(encoding='utf-8'))
    for nombre, esperado in manifiesto['archivos'].items():
        real = _sha256_archivo(carpeta / nombre)
        assert real == esperado, f'Hash inválido para {nombre}'
    datos = np.load(carpeta / 'banco.npz', allow_pickle=False)
    banco = datos['banco']
    assert banco.dtype == np.float32 and banco.ndim == 2, f'banco inválido: {banco.dtype} {banco.shape}'
    print(f'  Verificación OK: banco {banco.shape} · hashes correctos · carga sin pickle')

resumen_exportacion = []
for categoria in CATEGORIAS_A_EXPORTAR:
    print(f'\n=== {categoria} ===')
    carpeta_roi = ROI_ROOT / categoria / 'train' / 'good'
    if not carpeta_roi.exists():
        print(f'No hay dataset ROI para {categoria}; construyéndolo...')
        build_roi_dataset(categoria)

    rutas = sorted(carpeta_roi.glob('*.png'))
    barajadas = rutas[:]
    random.Random(SEMILLA).shuffle(barajadas)
    n_val = max(10, int(len(rutas) * FRACCION_VALIDACION))
    rutas_val, rutas_prep = sorted(barajadas[:n_val]), sorted(barajadas[n_val:])
    print(f'Preparación: {len(rutas_prep)} imágenes · Validación: {len(rutas_val)}')

    instancia, banco = preparar_patchcore(rutas_prep)
    puntuaciones_val, tiempos_det = puntuar(instancia, rutas_val)
    umbral = float(np.percentile(puntuaciones_val, PERCENTIL_UMBRAL))

    calibracion = {
        'categoria': categoria,
        'umbral': round(umbral, 6),
        'percentil': PERCENTIL_UMBRAL,
        'n_validacion': len(rutas_val),
        'n_preparacion': len(rutas_prep),
        'n_parches': int(banco.shape[0]),
        'dim_parche': int(banco.shape[1]),
        'capas': ['layer2', 'layer3'],
        'tam_entrada': [IMAGESIZE, IMAGESIZE],
        'resize': RESIZE,
        'coreset_pct': 10,
        'regla_seleccion': {
            'area_min_pct': int(AREA_MIN_RATIO * 100),
            'area_max_pct': int(AREA_MAX_RATIO * 100),
            'rho': RHO_OBJETIVO,
            'pesos': {'iou': PESO_IOU, 'estabilidad': PESO_ESTABILIDAD,
                      'relleno': PESO_RELLENO, 'borde': -PESO_BORDE},
        },
        'semilla': SEMILLA,
        'torchvision': torchvision.__version__,
    }
    carpeta = exportar_artefactos(categoria, banco, calibracion)
    verificar_artefactos(carpeta)
    resumen_exportacion.append({
        'categoria': categoria, 'umbral': round(umbral, 4),
        'parches': banco.shape[0],
        'deteccion_p95_ms': round(float(np.percentile(tiempos_det, 95)), 0),
    })
    print(f'Umbral p{PERCENTIL_UMBRAL}: {umbral:.4f} · '
          f'detección por imagen p95: {np.percentile(tiempos_det, 95):.0f} ms')
    print(f'Artefactos en: {carpeta}')

print('\nResumen:')
for r in resumen_exportacion:
    print(' ', r)


## 12. Guardar resultados y artefactos en Drive

Al terminar: descargar `samcore_artefactos/<categoría>/` de Drive y copiar la
carpeta a `backend/artefactos/<categoría>/` de la aplicación. `/api/salud`
debe listarla en `categorias_con_artefactos`.


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

drive_out = Path('/content/drive/MyDrive/patchcore_results_final')
drive_out.mkdir(parents=True, exist_ok=True)

# Resultados de las corridas — nombres POR CATEGORÍA para poder consolidar
# las 10 sin que se pisen
for etiqueta, carpeta in [('baseline', BASELINE_OUT), ('sam_roi', ROI_OUT)]:
    csv_path = carpeta / 'results.csv'
    if csv_path.exists():
        destino = drive_out / f'results_{etiqueta}_{CATEGORY}.csv'
        destino.write_bytes(csv_path.read_bytes())
        print('CSV copiado:', destino)
    for npz in carpeta.rglob('anomaly_data.npz'):
        destino = drive_out / f'anomaly_data_{etiqueta}_{CATEGORY}.npz'
        destino.write_bytes(npz.read_bytes())
        print('Datos crudos copiados:', destino)
        break

# Registros por categoría (estado ROI, AUROC re-proyectado)
for patron in ['roi_estado_*.csv', 'auroc_reproyectado_*.csv']:
    for p in sorted(DATA_ROOT.glob(patron)):
        (drive_out / p.name).write_bytes(p.read_bytes())
        print('Copiado:', drive_out / p.name)

figuras = [
    '/content/dataset_samples.png',
    '/content/sam_validation_good.png',
    '/content/sam_validation_defect.png',
    '/content/sam_roi_comparison.png',
    f'/content/results_table_{CATEGORY}.png',
    f'/content/roc_curves_reales_{CATEGORY}.png',
]
for fig_path in figuras:
    p = Path(fig_path)
    if p.exists():
        (drive_out / p.name).write_bytes(p.read_bytes())
        print('Figura copiada:', drive_out / p.name)

# Artefactos para la aplicación
drive_artefactos = Path('/content/drive/MyDrive/samcore_artefactos')
if ARTEFACTOS_ROOT.exists():
    shutil.copytree(ARTEFACTOS_ROOT, drive_artefactos, dirs_exist_ok=True)
    print('Artefactos copiados a:', drive_artefactos)

print()
print('Listo. Resultados y artefactos en Drive.')


## 13. Exportar la galería cerrada para la aplicación web

La aplicación ofrece un subconjunto del conjunto de prueba de MVTec AD
(6 imágenes normales y 2 por tipo de defecto, por categoría con banco).
Esta celda empaqueta el subconjunto de la categoría en curso en
`galeria.zip`; se descomprime en `backend/galeria/` del repositorio de la
aplicación (ver su README), donde se acumula con las demás categorías.


In [ ]:
import shutil, zipfile
from pathlib import Path

CATEGORIAS_GALERIA = [CATEGORY]   # la categoría en curso; el zip se descomprime en backend/galeria/
NORMALES, POR_DEFECTO = 6, 2

salida = Path('/content/galeria')
shutil.rmtree(salida, ignore_errors=True)
for categoria in CATEGORIAS_GALERIA:
    test = MVTEC_ROOT / categoria / 'test'
    for tipo in sorted(p.name for p in test.iterdir() if p.is_dir()):
        n = NORMALES if tipo == 'good' else POR_DEFECTO
        for origen in sorted((test / tipo).glob('*.png'))[:n]:
            destino = salida / categoria / tipo / origen.name
            destino.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(origen, destino)

zip_path = Path('/content/galeria.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(salida.rglob('*.png')):
        z.write(p, p.relative_to(salida))
print('Imágenes:', sum(1 for _ in salida.rglob('*.png')), '· zip:', zip_path, f'({zip_path.stat().st_size / 2**20:.1f} MB)')

from google.colab import files
files.download(str(zip_path))
